In [17]:
# =====================================================================
# IMPORTS AND SETUP
# =====================================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML
import os, json, re, webbrowser
# Set Mapbox access token for satellite map (replace with your own token if needed)
px.set_mapbox_access_token('pk.eyJ1IjoiY2hyaWRkeXAiLCJhIjoiY2lxMnlsaGFiMDA4dXRubmtiZGthNzh1YiJ9.b9YxH3oO4km0fLH1zfLfg')


In [18]:
df = pd.read_csv('Electric_Vehicle_Population_Data.csv')

df.dropna(inplace=True)

df.to_csv('Electric_Vehicle_Population_Data.csv', index=False)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 278984 entries, 0 to 278983
Data columns (total 16 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   VIN (1-10)                                         278984 non-null  str    
 1   County                                             278984 non-null  str    
 2   City                                               278984 non-null  str    
 3   State                                              278984 non-null  str    
 4   Postal Code                                        278984 non-null  float64
 5   Model Year                                         278984 non-null  int64  
 6   Make                                               278984 non-null  str    
 7   Model                                              278984 non-null  str    
 8   Electric Vehicle Type                              278984 non-null  str    
 9   Clea

In [19]:
df.head()

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
0,JN1AZ0CP5C,Stevens,Colville,WA,99114.0,2012,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73.0,7.0,153331706,POINT (-117.90454 48.54657),AVISTA CORP,5.306595e+10
1,JTMABABA7P,Yakima,Yakima,WA,98903.0,2023,SUBARU,SOLTERRA,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,15.0,253586308,POINT (-120.71847 46.55029),PACIFICORP,5.307700e+10
2,1N4AZ1CP1J,King,Seattle,WA,98122.0,2018,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,151.0,37.0,333135022,POINT (-122.31009 47.60803),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10
3,5UX43EU09S,Kitsap,Poulsbo,WA,98370.0,2025,BMW,X5,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,40.0,23.0,267525737,POINT (-122.64681 47.73689),PUGET SOUND ENERGY INC,5.303594e+10
4,3C3CFFGE5F,Thurston,Yelm,WA,98597.0,2015,FIAT,500,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,87.0,2.0,474468501,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,5.306701e+10


In [20]:
df.isnull().sum()

VIN (1-10)                                           0
County                                               0
City                                                 0
State                                                0
Postal Code                                          0
Model Year                                           0
Make                                                 0
Model                                                0
Electric Vehicle Type                                0
Clean Alternative Fuel Vehicle (CAFV) Eligibility    0
Electric Range                                       0
Legislative District                                 0
DOL Vehicle ID                                       0
Vehicle Location                                     0
Electric Utility                                     0
2020 Census Tract                                    0
dtype: int64

In [21]:
# ╔══════════════════════════════════════════════════════════╗
# ║  1. LOAD & CLEAN DATA                                   ║
# ╚══════════════════════════════════════════════════════════╝
try:
    df = pd.read_csv('Electric_Vehicle_Population_Data.csv')
    df.rename(columns={'Electric Vehicle Type': 'EV Type'}, inplace=True)
    df = df.dropna(subset=['Make', 'Model Year', 'Electric Range'])
    df['Model Year']     = df['Model Year'].astype(int)
    df['Electric Range'] = pd.to_numeric(df['Electric Range'], errors='coerce').fillna(0)
    if 'Model' not in df.columns and 'Vehicle Model' in df.columns:
        df['Model'] = df['Vehicle Model']
    print(f"Loaded {len(df):,} records")
except Exception as e:
    print(f"CSV not found ({e}). Using demo data.")
    np.random.seed(42)
    MAKES  = ['TESLA','CHEVROLET','NISSAN','FORD','BMW','KIA','HYUNDAI',
              'VOLKSWAGEN','AUDI','RIVIAN','VOLVO','MERCEDES-BENZ','PORSCHE',
              'JEEP','CHRYSLER','TOYOTA','SUBARU','LINCOLN','POLESTAR','MINI']
    MODELS = {
        'TESLA':['MODEL 3','MODEL Y','MODEL S','MODEL X'],
        'CHEVROLET':['BOLT EV','BOLT EUV','SPARK EV'],
        'NISSAN':['LEAF','ARIYA'],'FORD':['MUSTANG MACH-E','F-150 LIGHTNING'],
        'BMW':['I3','I4','IX'],'KIA':['EV6','NIRO EV','EV9'],
        'HYUNDAI':['IONIQ 5','IONIQ 6','KONA EV'],'VOLKSWAGEN':['ID.4','E-GOLF'],
        'AUDI':['E-TRON','Q4 E-TRON'],'RIVIAN':['R1T','R1S'],
        'VOLVO':['XC40 RECHARGE','C40 RECHARGE'],'MERCEDES-BENZ':['EQS','EQE','EQB'],
        'PORSCHE':['TAYCAN','TAYCAN CROSS TURISMO'],'JEEP':['WRANGLER 4XE','GRAND CHEROKEE 4XE'],
        'CHRYSLER':['PACIFICA HYBRID'],'TOYOTA':['RAV4 PRIME','PRIUS PRIME'],
        'SUBARU':['SOLTERRA'],'LINCOLN':['AVIATOR PHEV','CORSAIR PHEV'],
        'POLESTAR':['POLESTAR 2','POLESTAR 3'],'MINI':['MINI ELECTRIC','MINI CLUBMAN SE']
    }
    PROBS = [.26,.13,.11,.09,.06,.06,.06,.05,.04,.03,.02,.02,.015,.01,.01,.01,.01,.005,.005,.005]
    PROBS = [p/sum(PROBS) for p in PROBS]
    n = 8000
    years  = np.random.choice(range(2012,2024),n,p=[.01,.01,.02,.03,.05,.07,.09,.12,.14,.16,.16,.14])
    mk     = np.random.choice(MAKES,n,p=PROBS)
    rng    = np.clip(np.random.normal(130,70,n)+(years-2012)*14,0,400).astype(int)
    evtype = np.random.choice(['Battery Electric Vehicle (BEV)','Plug-in Hybrid Electric Vehicle (PHEV)'],n,p=[.62,.38])
    county = np.random.choice(['King','Snohomish','Pierce','Clark','Thurston','Kitsap','Spokane','Whatcom','Benton','Yakima'],n)
    df = pd.DataFrame({
        'Make':mk,'Model Year':years,'Electric Range':rng,'EV Type':evtype,
        'Model':[np.random.choice(MODELS[m]) for m in mk],
        'County':county,'State':np.full(n,'WA'),
        'Vehicle Location':[f'POINT ({np.random.uniform(-125,-67):.5f} {np.random.uniform(25,49):.5f})' for _ in range(n)]
    })


Loaded 278,984 records


In [30]:
# ╔══════════════════════════════════════════════════════════╗
# ║  2. AGGREGATE                                           ║
# ╚══════════════════════════════════════════════════════════╝
make_counts20 = df['Make'].value_counts().nlargest(20).reset_index()
make_counts20.columns = ['Make','Count']

year_counts = df.groupby('Model Year').size().reset_index(name='Count').sort_values('Model Year')
yc_s = year_counts.copy(); yc_s['Cumulative'] = yc_s['Count'].cumsum()

ev_type_dist = df['EV Type'].value_counts().reset_index()
ev_type_dist.columns = ['EV Type','Count']

avg_range_year = df.groupby('Model Year')['Electric Range'].mean().round(1).reset_index()
avg_range_year.columns = ['Model Year','Avg Range']
max_range_year = df.groupby('Model Year')['Electric Range'].max().reset_index()
max_range_year.columns = ['Model Year','Max Range']

range_by_make = (df[df['Make'].isin(make_counts20['Make'])]
                  .groupby('Make')['Electric Range'].mean().round(1).sort_values().reset_index())
range_by_make.columns = ['Make','Avg Range']

top20_models = pd.DataFrame()
if 'Model' in df.columns:
    top20_models = df['Model'].value_counts().nlargest(20).reset_index()
    top20_models.columns = ['Model','Count']

top10_makes = make_counts20['Make'].head(10).tolist()
hm_df = (df[df['Make'].isin(top10_makes)]
           .groupby(['Make','Model Year']).size().reset_index(name='Count'))

bev_phev = df.groupby(['Model Year','EV Type']).size().reset_index(name='Count')

bins   = [0,50,100,150,200,250,300,400]
labels = ['0-50','51-100','101-150','151-200','201-250','251-300','300+']
df['Range Bucket'] = pd.cut(df['Electric Range'],bins=bins,labels=labels,right=True)
range_bucket = df['Range Bucket'].value_counts().reindex(labels).fillna(0).reset_index()
range_bucket.columns = ['Bucket','Count']

county_dist = pd.DataFrame()
if 'County' in df.columns:
    county_dist = df['County'].value_counts().nlargest(15).reset_index()
    county_dist.columns = ['County','Count']

top5_makes = make_counts20['Make'].head(5).tolist()
mky = (df[df['Make'].isin(top5_makes)]
        .groupby(['Model Year','Make']).size().reset_index(name='Count'))

# Map points
map_data = []
if 'Vehicle Location' in df.columns:
    pat = re.compile(r'POINT \(([^ ]+) ([^)]+)\)')
    for _, row in df.sample(min(6000,len(df)),random_state=7).iterrows():
        m = pat.match(str(row.get('Vehicle Location','')))
        if m:
            lon,lat = float(m.group(1)), float(m.group(2))
            if -180<lon<180 and -90<lat<90:
                map_data.append({
                    "lat":round(lat,5),"lon":round(lon,5),
                    "make":str(row.get('Make',''))[:20],
                    "model":str(row.get('Model',''))[:20],
                    "range":int(row.get('Electric Range',0)),
                    "year":int(row.get('Model Year',0)),
                    "ev":str(row.get('EV Type',''))[:5]
                })

yr_last = int(year_counts['Model Year'].max())
yr_prev = yr_last - 1
c_curr  = int(year_counts[year_counts['Model Year']==yr_last]['Count'].sum())
c_prev  = int(year_counts[year_counts['Model Year']==yr_prev]['Count'].sum()) if yr_prev in year_counts['Model Year'].values else 1

kpi = {
    "total":    int(len(df)),
    "top_make": str(make_counts20.iloc[0]['Make']),
    "avg_range":float(round(df['Electric Range'].mean(),1)),
    "bev_pct":  float(round(df['EV Type'].str.contains('BEV',na=False).sum()/len(df)*100,1)),
    "yr_min":   int(df['Model Year'].min()),
    "yr_max":   int(df['Model Year'].max()),
    "n_makes":  int(df['Make'].nunique()),
    "n_models": int(df['Model'].nunique()) if 'Model' in df.columns else 0,
    "yoy":      round((c_curr-c_prev)/max(c_prev,1)*100,1),
    "c_curr":   c_curr,
}

payload = {
    "make_counts":    make_counts20.to_dict('records'),
    "year_counts":    year_counts.to_dict('records'),
    "year_counts_cum":yc_s[['Model Year','Cumulative']].to_dict('records'),
    "ev_type_dist":   ev_type_dist.to_dict('records'),
    "avg_range_year": avg_range_year.to_dict('records'),
    "max_range_year": max_range_year.to_dict('records'),
    "range_by_make":  range_by_make.to_dict('records'),
    "top20_models":   top20_models.to_dict('records') if len(top20_models) else [],
    "heatmap":        hm_df.to_dict('records'),
    "bev_phev":       bev_phev.to_dict('records'),
    "range_bucket":   range_bucket.to_dict('records'),
    "county_dist":    county_dist.to_dict('records') if len(county_dist) else [],
    "mky":            mky.to_dict('records'),
    "scatter_samp":   df[['Model Year','Electric Range','Make','EV Type']].sample(min(5000,len(df)),random_state=3).to_dict('records'),
    "map_data":       map_data,
    "kpi":            kpi,
    "all_makes":      sorted(df['Make'].unique().tolist()),
}

# ╔══════════════════════════════════════════════════════════╗
# ║  3. HTML                                                ║
# ╚══════════════════════════════════════════════════════════╝
HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/><meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>EV Analytics · Advanced Dashboard</title>
<link rel="preconnect" href="https://fonts.googleapis.com"/>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Space+Grotesk:wght@400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet"/>
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
<style>
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{
  --bg:#03060f;--surf:#080f1e;--card:#0b1628;--card2:#0e1c33;
  --border:#162540;--border2:#1e3a5f;
  --cyan:#00d4ff;--green:#10b981;--amber:#f59e0b;
  --rose:#f43f5e;--violet:#8b5cf6;--orange:#fb923c;
  --sky:#38bdf8;--teal:#2dd4bf;
  --text:#e2e8f0;--text2:#94a3b8;--text3:#475569;
  --ff:'Space Grotesk',sans-serif;--fi:'Inter',sans-serif;--mono:'JetBrains Mono',monospace;
  --r:10px;--r2:14px;--sh:0 4px 24px rgba(0,0,0,.5);
}
html,body{height:100%;background:var(--bg);color:var(--text);font-family:var(--ff);overflow:hidden}
::-webkit-scrollbar{width:4px;height:4px}
::-webkit-scrollbar-track{background:transparent}
::-webkit-scrollbar-thumb{background:var(--border2);border-radius:4px}

#app{display:grid;grid-template-rows:56px 1fr;height:100vh;width:100vw}

/* TOPBAR */
#topbar{display:flex;align-items:center;gap:12px;padding:0 18px;
  background:rgba(3,6,15,.96);border-bottom:1px solid var(--border);
  backdrop-filter:blur(20px);z-index:200;position:relative;flex-shrink:0}
.logo{font-size:16px;font-weight:800;white-space:nowrap}
.logo em{color:var(--cyan);font-style:normal}
.nav{display:flex;gap:3px;flex:1;justify-content:center}
.nb{font-family:var(--fi);font-size:11.5px;font-weight:500;padding:5px 14px;border-radius:7px;
  border:1px solid transparent;cursor:pointer;transition:all .18s;background:transparent;
  color:var(--text2)}
.nb:hover{color:var(--text);background:rgba(255,255,255,.05)}
.nb.on{background:var(--cyan);color:#000;font-weight:700;box-shadow:0 0 14px rgba(0,212,255,.3)}
.badges{display:flex;gap:7px;align-items:center;flex-shrink:0}
.badge{font-family:var(--mono);font-size:10px;padding:3px 9px;border-radius:20px;white-space:nowrap}
.bc{background:rgba(0,212,255,.1);border:1px solid rgba(0,212,255,.3);color:var(--cyan)}
.bg{background:rgba(16,185,129,.1);border:1px solid rgba(16,185,129,.3);color:var(--green)}
.tglow{position:absolute;bottom:0;left:0;right:0;height:1px;
  background:linear-gradient(90deg,transparent,var(--cyan),transparent);opacity:.5}

/* BODY */
#body{display:grid;grid-template-columns:262px 1fr;overflow:hidden}

/* SIDEBAR */
#sidebar{background:var(--surf);border-right:1px solid var(--border);
  padding:14px 12px;overflow-y:auto;display:flex;flex-direction:column;gap:16px}
.sh{font-size:9.5px;font-weight:700;letter-spacing:1.8px;text-transform:uppercase;
  color:var(--text3);margin-bottom:8px}
.ctrl{display:flex;flex-direction:column;gap:5px;margin-bottom:3px}
.ctrl label{font-size:11px;color:var(--text2);font-weight:500;font-family:var(--fi)}
.ctrl select,.ctrl input[type=range]{
  width:100%;background:var(--card);border:1px solid var(--border2);color:var(--text);
  border-radius:7px;padding:6px 10px;font-size:11.5px;font-family:var(--fi);
  outline:none;cursor:pointer;transition:border-color .2s;appearance:none}
.ctrl select:focus,.ctrl select:hover{border-color:var(--cyan)}
.ctrl input[type=range]{padding:2px 0;accent-color:var(--cyan)}
.rv{display:flex;justify-content:space-between;font-size:10px;
  color:var(--text3);font-family:var(--mono);margin-top:1px}
.rv span{color:var(--cyan)}

/* KPIs */
.kgrid{display:grid;grid-template-columns:1fr 1fr;gap:7px}
.kpi{background:var(--card);border:1px solid var(--border);border-radius:var(--r);
  padding:10px 11px;cursor:default;transition:all .2s;position:relative;overflow:hidden}
.kpi::after{content:'';position:absolute;top:0;left:0;right:0;height:2px;
  background:linear-gradient(90deg,var(--ka,var(--cyan)),transparent)}
.kpi:hover{border-color:var(--ka,var(--cyan));transform:translateY(-1px)}
.ki{font-size:16px;margin-bottom:3px}
.kl{font-size:9px;font-weight:600;letter-spacing:.9px;text-transform:uppercase;
  color:var(--text3);margin-bottom:3px;font-family:var(--fi)}
.kv{font-size:17px;font-weight:700;color:var(--ka,var(--cyan));line-height:1}
.ks{font-size:9px;color:var(--text3);margin-top:2px;font-family:var(--mono)}

/* spark */
.spark{display:flex;align-items:flex-end;gap:2px;height:22px;margin-top:6px}
.sb{flex:1;background:var(--cyan);border-radius:1px 1px 0 0;min-width:2px;opacity:.6;transition:height .3s}

/* quick stats */
.srow{display:grid;grid-template-columns:1fr 1fr;gap:6px}
.smin{background:var(--card2);border-radius:7px;padding:8px 9px;border:1px solid var(--border)}
.smv{font-size:15px;font-weight:700;color:var(--text);font-family:var(--mono)}
.sml{font-size:9px;color:var(--text3);text-transform:uppercase;letter-spacing:.8px;margin-top:2px}

/* MAIN */
#main{overflow-y:auto;padding:14px;background:var(--bg)}
.tab-panel{display:none}
.tab-panel.on{display:grid}
#t0{grid-template-columns:repeat(3,1fr);gap:13px;grid-template-rows:350px 270px}
#t1{grid-template-columns:1fr 1fr;grid-template-rows:1fr 1fr;gap:13px;height:calc(100vh - 56px - 28px)}
#t2{grid-template-columns:1fr 1fr;grid-template-rows:1fr 1fr;gap:13px;height:calc(100vh - 56px - 28px)}
#t3{grid-template-columns:1fr 1fr;grid-template-rows:1fr 1fr;gap:13px;height:calc(100vh - 56px - 28px)}
#t4{grid-template-columns:1fr 300px;gap:13px;height:calc(100vh - 56px - 28px)}

/* chart card */
.cc{background:var(--card);border:1px solid var(--border);border-radius:var(--r2);overflow:hidden;display:flex;flex-direction:column;transition:border-color .22s,box-shadow .22s;min-height:200px}
.cc:hover{border-color:rgba(0,212,255,.3);box-shadow:0 0 0 1px rgba(0,212,255,.08),var(--sh)}
.cc.s2{grid-column:span 2}
.cc.s3{grid-column:span 3}
.cc-h{display:flex;align-items:center;justify-content:space-between;padding:9px 13px 7px;flex-shrink:0;border-bottom:1px solid rgba(22,37,64,.6)}
.ct{font-size:11px;font-weight:600;color:var(--text2);font-family:var(--fi)}
.cp{font-size:9px;padding:2px 7px;border-radius:10px;font-family:var(--mono);
  background:rgba(0,212,255,.09);color:var(--cyan);border:1px solid rgba(0,212,255,.18)}
.ch{flex:1;min-height:0;overflow:hidden}

/* insights */
#ins{background:var(--surf);border:1px solid var(--border);border-radius:var(--r2);
  padding:14px;overflow-y:auto;display:flex;flex-direction:column;gap:11px}
.ib{background:var(--card);border:1px solid var(--border);border-radius:var(--r);
  padding:12px;border-left:3px solid var(--amber)}
.it{font-size:9.5px;font-weight:700;letter-spacing:1px;text-transform:uppercase;
  color:var(--amber);margin-bottom:7px}
.ir{font-size:11px;color:var(--text2);line-height:1.7;display:flex;gap:5px;
  align-items:flex-start;font-family:var(--fi)}
.ir::before{content:'›';color:var(--cyan);flex-shrink:0;font-weight:700}

/* loading */
#loading{position:fixed;inset:0;background:var(--bg);z-index:9999;
  display:flex;flex-direction:column;align-items:center;justify-content:center;gap:18px}
.ring{width:44px;height:44px;border:3px solid var(--border2);
  border-top-color:var(--cyan);border-radius:50%;animation:spin .75s linear infinite}
.lt{font-size:12px;color:var(--text3);font-family:var(--mono)}
@keyframes spin{to{transform:rotate(360deg)}}
@keyframes fadeU{from{opacity:0;transform:translateY(5px)}to{opacity:1;transform:none}}
.ib{animation:fadeU .4s ease both}
</style>
</head>
<body>
<div id="loading"><div class="ring"></div><div class="lt">Crunching EV data…</div></div>

<div id="app" style="opacity:0;transition:opacity .5s">

<div id="topbar">
  <div class="logo">⚡ EV <em>Analytics</em></div>
  <nav class="nav">
    <button class="nb on"  onclick="tab(0)">Overview</button>
    <button class="nb"     onclick="tab(1)">Growth &amp; Trends</button>
    <button class="nb"     onclick="tab(2)">Range &amp; Tech</button>
    <button class="nb"     onclick="tab(3)">Market Deep-Dive</button>
    <button class="nb"     onclick="tab(4)">Geo Map</button>
  </nav>
  <div class="badges">
    <span class="badge bc" id="b-total">— vehicles</span>
    <span class="badge bg" id="b-yoy">— YoY</span>
  </div>
  <div class="tglow"></div>
</div>

<div id="body">

<div id="sidebar">
  <div>
    <div class="sh">🎛 Filters</div>
    <div class="ctrl"><label>Manufacturer</label>
      <select id="f-make"><option value="All">All Manufacturers</option></select></div>
    <div class="ctrl"><label>EV Type</label>
      <select id="f-evtype">
        <option value="All">All Types</option>
        <option value="BEV">Battery Electric (BEV)</option>
        <option value="PHEV">Plug-in Hybrid (PHEV)</option>
      </select></div>
    <div class="ctrl">
      <label>Top N Makes: <span id="v-topn" style="color:var(--cyan)">20</span></label>
      <input type="range" id="f-topn" min="5" max="20" value="20"/>
      <div class="rv"><span>5</span><span>20</span></div>
    </div>
    <div class="ctrl">
      <label>Year From: <span id="v-yrmin" style="color:var(--cyan)">—</span></label>
      <input type="range" id="f-yrmin" min="2010" max="2024" value="2010"/>
    </div>
    <div class="ctrl">
      <label>Year To: <span id="v-yrmax" style="color:var(--cyan)">—</span></label>
      <input type="range" id="f-yrmax" min="2010" max="2024" value="2024"/>
    </div>
    <div class="ctrl">
      <label>Min Range (mi): <span id="v-rngmin" style="color:var(--cyan)">0</span></label>
      <input type="range" id="f-rngmin" min="0" max="400" step="10" value="0"/>
    </div>
  </div>

  <div>
    <div class="sh">📊 Live Metrics</div>
    <div class="kgrid">
      <div class="kpi" style="--ka:var(--cyan)"><div class="ki">🚗</div><div class="kl">Total EVs</div><div class="kv" id="k-total">—</div></div>
      <div class="kpi" style="--ka:var(--amber)"><div class="ki">🏆</div><div class="kl">Top Make</div><div class="kv" id="k-top" style="font-size:12px;color:var(--amber)">—</div></div>
      <div class="kpi" style="--ka:var(--green)"><div class="ki">⚡</div><div class="kl">Avg Range</div><div class="kv" id="k-range" style="color:var(--green)">—</div><div class="ks">miles</div></div>
      <div class="kpi" style="--ka:var(--violet)"><div class="ki">📈</div><div class="kl">YoY Growth</div><div class="kv" id="k-yoy" style="color:var(--violet)">—</div></div>
      <div class="kpi" style="--ka:var(--sky)"><div class="ki">🔋</div><div class="kl">BEV Share</div><div class="kv" id="k-bev" style="color:var(--sky)">—</div></div>
      <div class="kpi" style="--ka:var(--teal)"><div class="ki">🏭</div><div class="kl">Brands</div><div class="kv" id="k-makes" style="color:var(--teal)">—</div></div>
    </div>
    <div style="margin-top:10px"><div class="sh">Registration Spark</div><div class="spark" id="spark"></div></div>
  </div>

  <div>
    <div class="sh">Quick Stats</div>
    <div class="srow">
      <div class="smin"><div class="smv" id="qs-max">—</div><div class="sml">Max Range</div></div>
      <div class="smin"><div class="smv" id="qs-med">—</div><div class="sml">Median Range</div></div>
    </div>
  </div>
</div><!-- /sidebar -->

<div id="main">

  <!-- TAB 0: Overview -->
  <div id="t0" class="tab-panel on">
    <div class="cc"><div class="cc-h"><span class="ct">Top Manufacturers</span><span class="cp">RANKED</span></div><div id="c0" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">EV Type Hierarchy · Sunburst</span><span class="cp">DRILL DOWN</span></div><div id="c1" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">Market Share · Treemap</span><span class="cp">TOP 10</span></div><div id="c2" class="ch"></div></div>
    <div class="cc s3"><div class="cc-h"><span class="ct">Registration Timeline · Annual + Cumulative</span><span class="cp">DUAL AXIS</span></div><div id="c3" class="ch"></div></div>
  </div>

  <!-- TAB 1: Growth & Trends -->
  <div id="t1" class="tab-panel">
    <div class="cc"><div class="cc-h"><span class="ct">BEV vs PHEV · Stacked Area</span><span class="cp">BY YEAR</span></div><div id="c4" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">Top 5 Makes Over Time</span><span class="cp">MARKET RACE</span></div><div id="c5" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">Range Evolution · Avg &amp; Max</span><span class="cp">DUAL AXIS</span></div><div id="c6" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">Annual Registrations · Waterfall</span><span class="cp">DELTA VIEW</span></div><div id="c7" class="ch"></div></div>
  </div>

  <!-- TAB 2: Range & Tech -->
  <div id="t2" class="tab-panel">
    <div class="cc"><div class="cc-h"><span class="ct">Range Distribution · Violin</span><span class="cp">BY EV TYPE</span></div><div id="c8" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">Range × Year · Density Contour</span><span class="cp">HEATMAP</span></div><div id="c9" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">Volume YoY · Gauge</span><span class="cp">CURRENT YEAR</span></div><div id="c10" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">Avg Range per Make · Lollipop</span><span class="cp">RANKED</span></div><div id="c11" class="ch"></div></div>
  </div>

  <!-- TAB 3: Market Deep-Dive -->
  <div id="t3" class="tab-panel">
    <div class="cc"><div class="cc-h"><span class="ct">Top 20 Models · Treemap</span><span class="cp">CLICK EXPLORE</span></div><div id="c12" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">Range Spread by Make · Box Plot</span><span class="cp">DISTRIBUTION</span></div><div id="c13" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">Range vs Year · Scatter</span><span class="cp">SAMPLED</span></div><div id="c14" class="ch"></div></div>
    <div class="cc"><div class="cc-h"><span class="ct">County Distribution · Bubble</span><span class="cp">GEO SPLIT</span></div><div id="c15" class="ch"></div></div>
  </div>

  <!-- TAB 4: Geo Map -->
  <div id="t4" class="tab-panel">
    <div class="cc">
      <div class="cc-h" style="pointer-events:all;z-index:20">
        <span class="ct">EV Locations · Interactive Map</span>
        <div style="display:flex;gap:6px;pointer-events:all">
          <select id="map-color" onchange="buildMap()" style="font-size:11px;padding:4px 8px;border-radius:6px;background:#fff;border:1px solid #ccc;color:#333;font-family:sans-serif">
            <option value="range">Color: Range</option>
            <option value="year">Color: Year</option>
          </select>
          <select id="map-style" onchange="buildMap()" style="font-size:11px;padding:4px 8px;border-radius:6px;background:#fff;border:1px solid #ccc;color:#333;font-family:sans-serif">
            <option value="open-street-map">Street (Light)</option>
            <option value="carto-positron">Carto Light</option>
            <option value="carto-darkmatter">Carto Dark</option>
            <option value="stamen-terrain">Terrain</option>
          </select>
        </div>
      </div>
      <div id="c-map" class="ch" style="position:relative"></div>
    </div>
    <div id="ins">
      <div class="sh">Key Insights</div>
      <div class="ib">
        <div class="it">📊 Fleet Summary</div>
        <div class="ir" id="i-total">—</div>
        <div class="ir" id="i-top">—</div>
        <div class="ir" id="i-range">—</div>
        <div class="ir" id="i-yoy">—</div>
        <div class="ir" id="i-bev">—</div>
      </div>
      <div class="ib" style="border-left-color:var(--violet)">
        <div class="it" style="color:var(--violet)">💡 Industry Signals</div>
        <div class="ir">Battery range improving ~14 mi/year since 2012.</div>
        <div class="ir">BEV share exceeds 60% — PHEV serves as transition tech.</div>
        <div class="ir">Infrastructure gaps remain the #1 adoption bottleneck.</div>
        <div class="ir">Policy incentives drive 2× faster uptake in supported regions.</div>
        <div class="ir">New entrants (Rivian, Polestar) disrupting legacy OEM share.</div>
      </div>
      <div class="ib" style="border-left-color:var(--green)">
        <div class="it" style="color:var(--green)">🗺 Map Guide</div>
        <div class="ir">Each dot = 1 registered EV. Scroll/pinch to zoom.</div>
        <div class="ir">Color = selected metric (range or model year).</div>
        <div class="ir">Hover any point for make, model, range &amp; year.</div>
        <div class="ir">Switch map style &amp; color above. Export via camera icon.</div>
      </div>
    </div>
  </div>

</div><!-- /main -->
</div><!-- /body -->
</div><!-- /app -->

<script>
const D = __DATA__;

const T={
  bg:'#03060f',card:'#0b1628',card2:'#0e1c33',border:'#162540',border2:'#1e3a5f',
  cyan:'#00d4ff',green:'#10b981',amber:'#f59e0b',rose:'#f43f5e',violet:'#8b5cf6',
  orange:'#fb923c',sky:'#38bdf8',teal:'#2dd4bf',text:'#e2e8f0',muted:'#64748b',
  pal:['#00d4ff','#10b981','#f59e0b','#f43f5e','#8b5cf6','#fb923c',
       '#38bdf8','#2dd4bf','#a78bfa','#fbbf24','#34d399','#fb7185',
       '#60a5fa','#4ade80','#facc15','#e879f9','#22d3ee','#f97316','#84cc16','#ec4899'],
};
const BL={
  paper_bgcolor:T.card,plot_bgcolor:T.card,
  font:{color:T.text,family:"'Inter',sans-serif",size:11},
  margin:{l:44,r:14,t:6,b:36},
  hoverlabel:{bgcolor:'#0a1628',font:{size:12,family:"'Inter',sans-serif"},bordercolor:T.cyan},
  xaxis:{gridcolor:'#162540',linecolor:'#162540',tickfont:{size:10},zeroline:false},
  yaxis:{gridcolor:'#162540',linecolor:'#162540',tickfont:{size:10},zeroline:false},
  legend:{bgcolor:'transparent',font:{size:10},bordercolor:'transparent'},
  colorway:T.pal,
};
const CFG={displayModeBar:true,responsive:true,
  modeBarButtonsToRemove:['pan2d','lasso2d','select2d','autoScale2d'],
  toImageButtonOptions:{format:'png',filename:'ev_chart',scale:2}};
function lay(title,extra={}){
  return Object.assign({},BL,
    {title:{text:'',font:{size:11.5,color:T.muted,family:"'Inter',sans-serif"},
            x:.01,xanchor:'left',pad:{t:2}}},extra);
}
function layNoAxes(extra={}){
  // For charts that don't use cartesian axes (treemap, sunburst, pie, funnel)
  return {paper_bgcolor:T.card,plot_bgcolor:T.card,
    font:{color:T.text,family:"'Inter',sans-serif",size:11},
    margin:{l:4,r:4,t:6,b:4},
    hoverlabel:{bgcolor:'#0a1628',font:{size:12,family:"'Inter',sans-serif"},bordercolor:T.cyan},
    ...extra};
}

let S={make:'All',evtype:'All',topN:20,yrMin:null,yrMax:null,rngMin:0};
let curTab=0, mapBuilt=false;

function fMake(r){ return S.make==='All'||r.Make===S.make; }
function fEv(r){ return S.evtype==='All'||(r['EV Type']||'').includes(S.evtype); }
function fYr(r){ const y=r['Model Year'];return(!S.yrMin||y>=S.yrMin)&&(!S.yrMax||y<=S.yrMax); }
function fRng(r){ return (r['Electric Range']||0)>=S.rngMin; }
function fAll(r){ return fMake(r)&&fEv(r)&&fYr(r)&&fRng(r); }
function fSc(){ return D.scatter_samp.filter(fAll); }
function fYC(){ return D.year_counts.filter(r=>(!S.yrMin||r['Model Year']>=S.yrMin)&&(!S.yrMax||r['Model Year']<=S.yrMax)); }
function fMC(){
  let mc=D.make_counts;
  if(S.make!=='All') mc=mc.filter(r=>r.Make===S.make);
  return mc.slice(0,S.topN);
}

function updateKPIs(){
  const sc=fSc(), mc=fMC(), yc=fYC();
  const total=mc.reduce((s,r)=>s+r.Count,0);
  const top=mc.length?mc[0].Make:'—';
  const avgR=sc.length?(sc.reduce((s,r)=>s+r['Electric Range'],0)/sc.length).toFixed(1):'—';
  const maxR=sc.length?Math.max(...sc.map(r=>r['Electric Range'])):'—';
  const srt=[...sc].map(r=>r['Electric Range']).sort((a,b)=>a-b);
  const medR=srt.length?srt[Math.floor(srt.length/2)]:'—';
  const bevN=sc.filter(r=>(r['EV Type']||'').includes('BEV')).length;
  const bevP=sc.length?(bevN/sc.length*100).toFixed(0)+'%':'—';
  const ym=yc.length?Math.max(...yc.map(r=>r['Model Year'])):D.kpi.yr_max;
  const yp=ym-1;
  const cC=yc.find(r=>r['Model Year']===ym)?.Count||0;
  const cP=yc.find(r=>r['Model Year']===yp)?.Count||0;
  const yoy=cP?((cC-cP)/cP*100).toFixed(1):'—';

  document.getElementById('k-total').textContent=total.toLocaleString();
  document.getElementById('k-top').textContent=top;
  document.getElementById('k-range').textContent=avgR;
  const ye=document.getElementById('k-yoy');
  ye.textContent=yoy!=='—'?yoy+'%':'—';
  ye.style.color=parseFloat(yoy)>=0?T.green:T.rose;
  document.getElementById('k-bev').textContent=bevP;
  document.getElementById('k-makes').textContent=mc.length;
  document.getElementById('b-total').textContent=total.toLocaleString()+' EVs';
  document.getElementById('b-yoy').textContent=yoy!=='—'?(parseFloat(yoy)>=0?'+':'')+yoy+'% YoY':'—';
  document.getElementById('qs-max').textContent=maxR+'mi';
  document.getElementById('qs-med').textContent=medR+'mi';

  const sp=document.getElementById('spark');sp.innerHTML='';
  const ycs=yc.slice(-14);const mx=Math.max(...ycs.map(r=>r.Count),1);
  ycs.forEach(r=>{const b=document.createElement('div');b.className='sb';
    b.style.height=(r.Count/mx*100)+'%';b.title=`${r['Model Year']}: ${r.Count.toLocaleString()}`;
    sp.appendChild(b);});

  document.getElementById('i-total').textContent=`${total.toLocaleString()} EVs in current filter.`;
  document.getElementById('i-top').textContent=`${top} leads market share.`;
  document.getElementById('i-range').textContent=`Avg range ${avgR} mi — max ${maxR} mi.`;
  document.getElementById('i-yoy').textContent=`YoY growth: ${yoy}% (${yp}→${ym}).`;
  document.getElementById('i-bev').textContent=`BEV share: ${bevP} of filtered fleet.`;
}

/* ── CHARTS ── */
function c0(){
  const mc=[...fMC()].reverse();
  Plotly.react('c0',[{
    type:'bar',orientation:'h',
    x:mc.map(r=>r.Count),y:mc.map(r=>r.Make),
    marker:{color:mc.map((_,i)=>T.pal[i%T.pal.length]),line:{color:T.card,width:1}},
    text:mc.map(r=>r.Count.toLocaleString()),
    textposition:'outside',textfont:{size:9,color:T.muted},
    hovertemplate:'<b>%{y}</b><br>%{x:,} vehicles<extra></extra>',
  }],lay('Top Manufacturers',{margin:{l:120,r:60,t:6,b:20},bargap:.28,
    xaxis:{...BL.xaxis,showgrid:true},yaxis:{...BL.yaxis,tickfont:{size:10}}}),CFG);
}

function c1(){
  const sc=fSc();
  const counts={};
  sc.forEach(r=>{
    const t=(r['EV Type']||'').includes('BEV')?'BEV':'PHEV';
    const k=r.Make+'|'+t;counts[k]=(counts[k]||0)+1;
  });
  const labels=['All EVs'],parents=[''],vals=[sc.length],typeMap={};
  Object.entries(counts).forEach(([k,v])=>{
    const [make,tp]=k.split('|');
    if(!typeMap[tp]){typeMap[tp]=0;labels.push(tp);parents.push('All EVs');vals.push(0);}
    typeMap[tp]+=v;labels.push(make+' '+tp);parents.push(tp);vals.push(v);
  });
  Object.entries(typeMap).forEach(([tp,v])=>{const i=labels.indexOf(tp);if(i>-1)vals[i]=v;});
  Plotly.react('c1',[{
    type:'sunburst',labels,parents,values:vals,branchvalues:'total',
    textfont:{size:9},
    marker:{colors:labels.map((_,i)=>T.pal[i%T.pal.length]),line:{color:T.card,width:2}},
    hovertemplate:'<b>%{label}</b><br>%{value:,}<br>%{percentParent:.1%} of parent<extra></extra>',
  }],lay('EV Type Hierarchy',{margin:{l:8,r:8,t:6,b:8}}),CFG);
}

function c2(){
  const mc=fMC().slice(0,10);
  const rootTotal = mc.reduce((s,r)=>s+r.Count,0);
  Plotly.react('c2',[{
    type:'treemap',
    labels:['Fleet',...mc.map(r=>r.Make)],
    parents:['',...mc.map(()=>'Fleet')],
    values:[rootTotal,...mc.map(r=>r.Count)],
    branchvalues:'total',
    texttemplate:'<b>%{label}</b><br>%{value:,}<br>%{percentRoot:.0%}',
    textfont:{size:11},
    marker:{colors:['rgba(0,0,0,0)',...mc.map((_,i)=>T.pal[i%T.pal.length])],
            line:{color:'rgba(3,6,15,.6)',width:2},showscale:false},
    hovertemplate:'<b>%{label}</b><br>%{value:,}<br>%{percentParent:.1%}<extra></extra>',
    tiling:{packing:'squarify'},
  }],layNoAxes({autosize:true}),CFG);
}

function c3(){
  const yc=fYC();const cum=[];let s=0;
  yc.forEach(r=>{s+=r.Count;cum.push(s);});
  Plotly.react('c3',[
    {type:'scatter',name:'Annual',mode:'lines+markers',
     x:yc.map(r=>r['Model Year']),y:yc.map(r=>r.Count),
     fill:'tozeroy',fillcolor:'rgba(0,212,255,.09)',
     line:{color:T.cyan,width:2.5},marker:{size:7,color:T.card,line:{width:2,color:T.cyan}},
     hovertemplate:'<b>%{x}</b> — New: %{y:,}<extra></extra>'},
    {type:'scatter',name:'Cumulative',mode:'lines',yaxis:'y2',
     x:yc.map(r=>r['Model Year']),y:cum,
     line:{color:T.amber,width:2,dash:'dot'},
     hovertemplate:'<b>%{x}</b> — Total: %{y:,}<extra></extra>'},
  ],lay('Annual vs Cumulative Registrations',{
    margin:{l:54,r:54,t:6,b:36},hovermode:'x unified',
    legend:{x:.01,y:.99,orientation:'h'},
    yaxis:{...BL.yaxis,title:{text:'Annual',font:{size:10,color:T.cyan}}},
    yaxis2:{overlaying:'y',side:'right',title:{text:'Cumulative',font:{size:10,color:T.amber}},
            gridcolor:'transparent',linecolor:'transparent',tickfont:{size:10}},
  }),CFG);
}

function c4(){
  const bev=D.bev_phev.filter(r=>r['EV Type'].includes('BEV')).filter(r=>fYr(r));
  const phev=D.bev_phev.filter(r=>r['EV Type'].includes('PHEV')).filter(r=>fYr(r));
  Plotly.react('c4',[
    {type:'scatter',name:'BEV',mode:'lines',
     x:bev.map(r=>r['Model Year']),y:bev.map(r=>r.Count),
     stackgroup:'one',fillcolor:'rgba(0,212,255,.6)',line:{color:T.cyan,width:0},
     hovertemplate:'BEV %{x}: %{y:,}<extra></extra>'},
    {type:'scatter',name:'PHEV',mode:'lines',
     x:phev.map(r=>r['Model Year']),y:phev.map(r=>r.Count),
     stackgroup:'one',fillcolor:'rgba(245,158,11,.5)',line:{color:T.amber,width:0},
     hovertemplate:'PHEV %{x}: %{y:,}<extra></extra>'},
  ],lay('BEV vs PHEV Stacked Area',{
    legend:{x:.01,y:.99,orientation:'h'},hovermode:'x unified',
    margin:{l:44,r:14,t:6,b:36},
  }),CFG);
}

function c5(){
  const top5=D.make_counts.slice(0,5).map(r=>r.Make);
  const traces=top5.map((mk,i)=>({
    type:'scatter',name:mk,mode:'lines+markers',
    x:D.mky.filter(r=>r.Make===mk&&fYr(r)).map(r=>r['Model Year']),
    y:D.mky.filter(r=>r.Make===mk&&fYr(r)).map(r=>r.Count),
    line:{color:T.pal[i],width:2.5},marker:{size:5,color:T.pal[i]},
    hovertemplate:`<b>${mk}</b> %{x}: %{y:,}<extra></extra>`,
  }));
  Plotly.react('c5',traces,lay('Top 5 Makes Over Time',{
    hovermode:'x unified',legend:{x:.01,y:.99,orientation:'h'},
    margin:{l:44,r:14,t:6,b:36},
  }),CFG);
}

function c6(){
  const ar=D.avg_range_year.filter(r=>fYr(r));
  const mr=D.max_range_year.filter(r=>fYr(r));
  Plotly.react('c6',[
    {type:'scatter',name:'Avg Range',mode:'lines+markers',
     x:ar.map(r=>r['Model Year']),y:ar.map(r=>r['Avg Range']),
     line:{color:T.green,width:3},fill:'tozeroy',fillcolor:'rgba(16,185,129,.1)',
     marker:{size:7,color:T.card,line:{width:2,color:T.green}},
     hovertemplate:'Avg: %{y:.0f} mi (%{x})<extra></extra>'},
    {type:'scatter',name:'Max Range',mode:'lines',yaxis:'y2',
     x:mr.map(r=>r['Model Year']),y:mr.map(r=>r['Max Range']),
     line:{color:T.violet,width:2,dash:'dashdot'},
     hovertemplate:'Max: %{y:.0f} mi (%{x})<extra></extra>'},
  ],lay('Range Evolution — Avg & Max',{
    yaxis:{...BL.yaxis,title:{text:'Avg (mi)',font:{size:10,color:T.green}}},
    yaxis2:{overlaying:'y',side:'right',title:{text:'Max (mi)',font:{size:10,color:T.violet}},
            gridcolor:'transparent',linecolor:'transparent',tickfont:{size:10}},
    legend:{x:.01,y:.99,orientation:'h'},hovermode:'x unified',
    margin:{l:54,r:54,t:6,b:36},
  }),CFG);
}

function c7(){
  const yc=fYC();
  const deltas=yc.map((r,i)=>i===0?r.Count:r.Count-yc[i-1].Count);
  Plotly.react('c7',[{
    type:'waterfall',
    x:yc.map(r=>r['Model Year']),y:deltas,
    measure:yc.map((_,i)=>i===0?'absolute':'relative'),
    connector:{line:{color:T.border2,width:1.5}},
    increasing:{marker:{color:T.green,line:{color:T.card,width:1}}},
    decreasing:{marker:{color:T.rose,line:{color:T.card,width:1}}},
    totals:{marker:{color:T.cyan}},
    hovertemplate:'<b>%{x}</b><br>Δ %{y:+,}<extra></extra>',
  }],lay('Annual Registration Delta — Waterfall',{margin:{l:44,r:14,t:6,b:36}}),CFG);
}

function c8(){
  const sc=fSc();
  const bev=sc.filter(r=>(r['EV Type']||'').includes('BEV')).map(r=>r['Electric Range']);
  const phev=sc.filter(r=>(r['EV Type']||'').includes('PHEV')).map(r=>r['Electric Range']);
  Plotly.react('c8',[
    {type:'violin',y:bev,name:'BEV',box:{visible:true},meanline:{visible:true},
     line:{color:T.cyan},fillcolor:'rgba(0,212,255,.13)',
     hovertemplate:'BEV: %{y:.0f} mi<extra></extra>'},
    {type:'violin',y:phev,name:'PHEV',box:{visible:true},meanline:{visible:true},
     line:{color:T.amber},fillcolor:'rgba(245,158,11,.13)',
     hovertemplate:'PHEV: %{y:.0f} mi<extra></extra>'},
  ],lay('Range Distribution by EV Type',{
    violingap:.3,violinmode:'overlay',legend:{x:.01,y:.99},
    margin:{l:44,r:14,t:6,b:36},
  }),CFG);
}

function c9(){
  const sc=fSc();
  Plotly.react('c9',[{
    type:'histogram2dcontour',
    x:sc.map(r=>r['Model Year']),y:sc.map(r=>r['Electric Range']),
    colorscale:'Plasma',showscale:true,
    contours:{showlabels:true,labelfont:{size:9,color:T.text}},
    hovertemplate:'Year: %{x}<br>Range: %{y:.0f} mi<br>Density: %{z}<extra></extra>',
    colorbar:{tickfont:{size:9,color:T.text},
      title:{text:'Density',font:{size:10,color:T.muted}},x:1.01,len:.8},
  }],lay('Range × Year Density Contour',{
    margin:{l:54,r:64,t:6,b:44},
    xaxis:{...BL.xaxis,title:{text:'Model Year',font:{size:10}}},
    yaxis:{...BL.yaxis,title:{text:'Electric Range (mi)',font:{size:10}}},
  }),CFG);
}


function c10(){
  const rb=D.range_bucket;
  const total=rb.reduce((s,r)=>s+r.Count,0)||1;
  const labels=['0-50','51-100','101-150','151-200','201-250','251-300','300+'];
  const pcts=labels.map(l=>{
    const row=rb.find(r=>r.Bucket===l);
    return row?Math.round(row.Count/total*100):0;
  });
  const maxPct=Math.max(...pcts,1);
  const traces=labels.map((lbl,i)=>({
    type:'indicator',
    mode:'gauge+number',
    value:pcts[i],
    title:{text:lbl+' mi',font:{size:9.5,color:T.pal[i%T.pal.length]}},
    number:{suffix:'%',font:{size:13,color:T.pal[i%T.pal.length]}},
    gauge:{
      axis:{range:[0,maxPct*1.1],tickfont:{size:7,color:T.muted},nticks:3},
      bar:{color:T.pal[i%T.pal.length],thickness:.38},
      bgcolor:T.card2,
      borderwidth:1,bordercolor:T.border2,
      steps:[{range:[0,maxPct*.5],color:'rgba(255,255,255,.03)'}],
      threshold:{line:{color:T.pal[i%T.pal.length],width:2},thickness:.75,value:pcts[i]*.95},
    },
    domain:{row:Math.floor(i/4),column:i%4},
  }));
  Plotly.react('c10',traces,{
    paper_bgcolor:T.card,plot_bgcolor:T.card,
    font:{color:T.text,family:"'Inter',sans-serif",size:11},
    margin:{l:10,r:10,t:14,b:8},
    grid:{rows:2,columns:4,pattern:'independent'},
    hoverlabel:{bgcolor:'#0a1628',font:{size:12},bordercolor:T.cyan},
    autosize:true,
  },CFG);
}

function c11(){
  const hm=D.heatmap.filter(r=>fYr(r));
  const makes=[...new Set(hm.map(r=>r.Make))];
  const years=[...new Set(hm.map(r=>r['Model Year']))].sort();
  const z=makes.map(mk=>years.map(yr=>{
    const row=hm.find(r=>r.Make===mk&&r['Model Year']===yr);return row?row.Count:0;
  }));
  Plotly.react('c11',[{
    type:'heatmap',z,x:years,y:makes,
    colorscale:'Viridis',showscale:true,
    hovertemplate:'<b>%{y}</b><br>Year: %{x}<br>Count: %{z:,}<extra></extra>',
    colorbar:{tickfont:{size:9,color:T.text},title:{text:'Count',font:{size:10,color:T.muted}},x:1.01,len:.8},
  }],lay('Make × Year Registration Heatmap',{
    margin:{l:130,r:64,t:6,b:44},
    xaxis:{...BL.xaxis,title:{text:'Model Year',font:{size:10}},type:'category'},
    yaxis:{...BL.yaxis,tickfont:{size:10}},
  }),CFG);
}
function c12(){
  const tm=D.top20_models.slice(0,20);
  if(!tm||!tm.length){
    document.getElementById('c12').innerHTML=
      '<div style="height:100%;display:flex;align-items:center;justify-content:center;color:'+T.muted+';font-size:13px">No model data</div>';
    return;
  }
  const rootTotal = tm.reduce((s,r)=>s+r.Count,0);
  Plotly.react('c12',[{
    type:'treemap',
    labels:['All Models',...tm.map(r=>r.Model)],
    parents:['',...tm.map(()=>'All Models')],
    values:[rootTotal,...tm.map(r=>r.Count)], // ← FIX: was 0
    branchvalues:'total',
    texttemplate:'<b>%{label}</b><br>%{value:,}',
    textfont:{size:10},
    marker:{colors:['rgba(0,0,0,0)',...tm.map((_,i)=>T.pal[i%T.pal.length])],
            line:{color:'rgba(3,6,15,.6)',width:2},showscale:false},
    hovertemplate:'<b>%{label}</b><br>%{value:,} vehicles<br>%{percentParent:.1%}<extra></extra>',
    tiling:{packing:'squarify'},
  }],layNoAxes({autosize:true}),CFG);
}

function c13(){
  // Range Spread by Make — Box Plot (like the screenshot)
  const sc=fSc();
  const makeList=[...new Set(sc.map(r=>r.Make))].sort();
  const traces=makeList.map((mk,i)=>{
    const vals=sc.filter(r=>r.Make===mk).map(r=>r['Electric Range']);
    return {
      type:'box',
      y:vals,
      name:mk,
      boxpoints:'outliers',
      marker:{color:T.pal[i%T.pal.length],size:4,opacity:.6,
              line:{color:T.card,width:1}},
      line:{color:T.pal[i%T.pal.length],width:1.5},
      fillcolor:T.pal[i%T.pal.length].replace(')',', .3)').replace('rgb','rgba'),
      hovertemplate:'<b>%{x}</b><br>Range: %{y:.0f} mi<extra></extra>',
    };
  });
  Plotly.react('c13',traces,lay('Range Spread by Make',{
    margin:{l:54,r:14,t:6,b:90},
    xaxis:{...BL.xaxis,tickangle:-35,tickfont:{size:9.5}},
    yaxis:{...BL.yaxis,title:{text:'Electric Range (mi)',font:{size:10}},zeroline:false},
    showlegend:false,
    boxmode:'group',
  }),CFG);
}

function c14(){
  // Range vs Year (Sampled) Scatter — like screenshot 4
  const sc=fSc();
  const makes=[...new Set(sc.map(r=>r.Make))].sort();
  const traces=makes.map((mk,i)=>{
    const pts=sc.filter(r=>r.Make===mk);
    return {
      type:'scatter',mode:'markers',name:mk,
      x:pts.map(r=>r['Model Year']),
      y:pts.map(r=>r['Electric Range']),
      marker:{
        color:T.pal[i%T.pal.length],
        size:6,opacity:.7,
        line:{color:'rgba(0,0,0,.2)',width:.5},
      },
      hovertemplate:'<b>%{text}</b><br>Year: %{x}<br>Range: %{y:.0f} mi<extra></extra>',
      text:pts.map(r=>r.Make),
      showlegend:makes.length<=12,
    };
  });
  Plotly.react('c14',traces,lay('Range vs Year — Sampled',{
    margin:{l:54,r:14,t:6,b:44},
    xaxis:{...BL.xaxis,title:{text:'Model Year',font:{size:10}},dtick:2,tickformat:'d'},
    yaxis:{...BL.yaxis,title:{text:'Electric Range (mi)',font:{size:10}},zeroline:false},
    legend:{x:1.01,y:.99,font:{size:9},xanchor:'left'},
    hovermode:'closest',
  }),CFG);
}
function c15(){
  const cd=D.county_dist;
  if(!cd||!cd.length){
    document.getElementById('c15').innerHTML=
      `<div style="height:100%;display:flex;align-items:center;justify-content:center;color:${T.muted};font-size:13px">No county data</div>`;
    return;
  }
  const max=Math.max(...cd.map(r=>r.Count));
  Plotly.react('c15',[{
    type:'scatter',mode:'markers+text',
    x:cd.map((_,i)=>i%5),y:cd.map((_,i)=>Math.floor(i/5)),
    text:cd.map(r=>r.County),textposition:'middle center',textfont:{size:10,color:'#fff',family:"'Inter',sans-serif"},
    marker:{size:cd.map(r=>Math.sqrt(r.Count/max)*75+20),
            color:cd.map((_,i)=>T.pal[i%T.pal.length]),opacity:.82,
            line:{color:T.card,width:1.5},sizemode:'diameter'},
    customdata:cd.map(r=>r.Count),
    hovertemplate:'<b>%{text}</b><br>%{customdata:,} vehicles<extra></extra>',
  }],lay('County Distribution — Bubble',{
    margin:{l:14,r:14,t:6,b:14},showlegend:false,
    xaxis:{...BL.xaxis,showgrid:false,showticklabels:false,zeroline:false},
    yaxis:{...BL.yaxis,showgrid:false,showticklabels:false,zeroline:false},
  }),CFG);
}

function buildMap(){
  const pts=D.map_data;
  if(!pts||!pts.length){
    document.getElementById('c-map').innerHTML=
      `<div style="height:100%;display:flex;align-items:center;justify-content:center;color:${T.muted};font-size:13px">No location data available</div>`;
    return;
  }
  const colorBy=document.getElementById('map-color')?.value||'range';
  const mapStyle=document.getElementById('map-style')?.value||'open-street-map';
  const vals=pts.map(p=>colorBy==='range'?p.range:p.year);
  const cmin=Math.min(...vals),cmax=Math.max(...vals);
  const scale=colorBy==='range'?'Plasma':'Turbo';
  const avgLat=pts.reduce((s,p)=>s+p.lat,0)/pts.length;
  const avgLon=pts.reduce((s,p)=>s+p.lon,0)/pts.length;

  Plotly.react('c-map',[{
    type:'scattermapbox',
    lat:pts.map(p=>p.lat),lon:pts.map(p=>p.lon),
    mode:'markers',
    marker:{size:6,color:vals,colorscale:scale,cmin,cmax,opacity:.8,
      colorbar:{title:{text:colorBy==='range'?'Range (mi)':'Year',font:{size:11,color:'#333'}},
        tickfont:{size:10,color:'#333'},bgcolor:'rgba(255,255,255,.9)',
        bordercolor:'#ccc',borderwidth:1,x:.01,xanchor:'left',len:.55,thickness:12}},
    customdata:pts.map(p=>[p.make,p.model,p.range,p.year,p.ev]),
    hovertemplate:
      '<b>%{customdata[0]}</b> %{customdata[1]}<br>'+
      'Range: %{customdata[2]} mi · Year: %{customdata[3]}<br>'+
      'Type: %{customdata[4]}<extra></extra>',
  }],{
    mapbox:{style:mapStyle,zoom:3,center:{lat:avgLat||38,lon:avgLon||-96}},
    margin:{l:0,r:0,t:0,b:0},
    paper_bgcolor:'rgba(0,0,0,0)',plot_bgcolor:'rgba(0,0,0,0)',
  },{...CFG,displayModeBar:true,
    modeBarButtonsToRemove:['select2d','lasso2d'],
    toImageButtonOptions:{format:'png',filename:'ev_map',scale:2}});
  mapBuilt=true;
}

function buildTab(){
  updateKPIs();
  if(curTab===0){c0();c1();c2();c3();}
  else if(curTab===1){c4();c5();c6();c7();}
  else if(curTab===2){c8();c9();c10();c11();}
  else if(curTab===3){c12();c13();c14();c15();}
  else if(curTab===4){buildMap();}
}

function tab(n){
  document.querySelectorAll('.tab-panel').forEach((p,i)=>p.classList.toggle('on',i===n));
  document.querySelectorAll('.nb').forEach((b,i)=>b.classList.toggle('on',i===n));
  curTab=n;
  setTimeout(()=>{buildTab();window.dispatchEvent(new Event('resize'));},60);
}

let raf=null;
function refresh(){
  if(raf)cancelAnimationFrame(raf);
  raf=requestAnimationFrame(()=>buildTab());
}

function initControls(){
  const sel=document.getElementById('f-make');
  D.all_makes.forEach(m=>{const o=document.createElement('option');o.value=o.textContent=m;sel.appendChild(o);});
  const years=D.year_counts.map(r=>r['Model Year']);
  const yMin=Math.min(...years),yMax=Math.max(...years);
  S.yrMin=yMin;S.yrMax=yMax;
  ['f-yrmin','f-yrmax'].forEach(id=>{
    const el=document.getElementById(id);el.min=yMin;el.max=yMax;
    el.value=id==='f-yrmin'?yMin:yMax;
  });
  document.getElementById('v-yrmin').textContent=yMin;
  document.getElementById('v-yrmax').textContent=yMax;

  sel.addEventListener('change',e=>{S.make=e.target.value;refresh();});
  document.getElementById('f-evtype').addEventListener('change',e=>{S.evtype=e.target.value;refresh();});
  document.getElementById('f-topn').addEventListener('input',e=>{
    S.topN=+e.target.value;document.getElementById('v-topn').textContent=e.target.value;refresh();});
  document.getElementById('f-yrmin').addEventListener('input',e=>{
    S.yrMin=+e.target.value;document.getElementById('v-yrmin').textContent=e.target.value;refresh();});
  document.getElementById('f-yrmax').addEventListener('input',e=>{
    S.yrMax=+e.target.value;document.getElementById('v-yrmax').textContent=e.target.value;refresh();});
  document.getElementById('f-rngmin').addEventListener('input',e=>{
    S.rngMin=+e.target.value;document.getElementById('v-rngmin').textContent=e.target.value;refresh();});
}

let rt;
window.addEventListener('resize',()=>{
  clearTimeout(rt);rt=setTimeout(()=>{
    document.querySelectorAll('.plotly-graph-div').forEach(el=>{
      try{Plotly.Plots.resize(el);}catch(e){}
    });
  },120);
});

window.addEventListener('load',()=>{
  initControls();buildTab();
  setTimeout(()=>{
    document.getElementById('loading').style.display='none';
    document.getElementById('app').style.opacity='1';
  },650);
});
</script>
</body></html>"""
# ╔══════════════════════════════════════════════════════════╗
# ║  4. INJECT & WRITE                                      ║
# ╚══════════════════════════════════════════════════════════╝
html_out = HTML.replace('__DATA__', json.dumps(payload))
out_path = os.path.join(os.getcwd(), 'ev_dashboard.html')
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(html_out)

print(f"Dashboard saved: {out_path}")
print("Opening browser…")
webbrowser.open('file://' + os.path.abspath(out_path))

try:
    from IPython.display import IFrame, display as ipyd
    ipyd(IFrame(src='ev_dashboard.html', width='100%', height='800'))
    print("Inline preview rendered above.")
except ImportError:
    pass

Dashboard saved: C:\Users\SAGAR\PYTHON TOOLS PROJECT\ev_dashboard.html
Opening browser…


Inline preview rendered above.
